In [3]:
import pandas as pd
import numpy as np
import re

print("1. Loading 97k dataset... (Wait ~30-60 seconds)")
file_path = r"C:\Users\ashis\OneDrive\Desktop\Analyst_Job_Market_Project_2026\1_Raw_Data\indian-job-market-dataset-2025.xlsx"
df = pd.read_excel(file_path)

print("2. Filtering for Analyst roles...")
df = df.dropna(subset=['title'])
df_analyst = df[df['title'].str.contains('Analyst', case=False, na=False)].copy()

print("3. Stripping HTML tags...")
def strip_html(text):
    if pd.isna(text): return ""
    return re.sub(r'<.*?>', '', str(text))
df_analyst['jobDescription'] = df_analyst['jobDescription'].apply(strip_html)

print("4. Normalizing Indian City names (RESTORED)...")
df_analyst['Clean_City'] = df_analyst['location'].str.split(r'[,/\(]').str[0].str.strip().str.title()
city_map = {
    'Bengaluru': 'Bangalore', 'Navi Mumbai': 'Mumbai', 'Thane': 'Mumbai',
    'Gurgaon': 'Gurugram', 'Noida': 'Delhi NCR', 'New Delhi': 'Delhi NCR',
    'Delhi': 'Delhi NCR', 'Faridabad': 'Delhi NCR', 'Secunderabad': 'Hyderabad',
    'Chennai': 'Chennai', 'Kolkata': 'Kolkata'
}
df_analyst['Clean_City'] = df_analyst['Clean_City'].replace(city_map)

print("5. Categorizing Industries (V2)...")
def get_industry_v2(company):
    c = str(company).lower()
    if any(x in c for x in ['bank', 'finance', 'capital', 'insurance', 'finserv', 'hdfc', 'icici', 'kotak', 'axis', 'sbi', 'bajaj', 'nbfc', 'credit', 'wealth', 'trading', 'securities']): 
        return 'BFSI'
    if any(x in c for x in ['tech', 'software', 'solutions', 'systems', 'info', 'wipro', 'infosys', 'tcs', 'hcl', 'cognizant', 'accenture', 'capgemini', 'mphasis', 'lti', 'ibm', 'microsoft', 'google', 'amazon', 'deloitte', 'data', 'analytics', 'cloud', 'digital']): 
        return 'IT/Tech'
    if any(x in c for x in ['tata', 'reliance', 'mahindra', 'larsen', 'l&t', 'steel', 'power', 'energy', 'jindal', 'adani', 'auto', 'motors', 'bosch', 'siemens', 'vedanta', 'maruti', 'construction', 'engineering']):
        return 'Manufacturing/Core'
    if any(x in c for x in ['staffing', 'services', 'manpower', 'hr', 'recruitment', 'consultancy', 'job', 'career', 'adecco', 'randstad', 'quess', 'teamlease', 'kelly']):
        return 'Staffing/HR Services'
    if any(x in c for x in ['flipkart', 'myntra', 'retail', 'ecommerce', 'swiggy', 'zomato', 'meesho', 'nykaa', 'bigbasket', 'delivery', 'logistics', 'express']): 
        return 'Ecommerce/Logistics'
    if any(x in c for x in ['pharma', 'health', 'medical', 'hospital', 'apollo', 'fortis', 'cipla', 'sun pharma', 'dr reddy', 'biotech', 'life sciences']): 
        return 'Healthcare'
    if any(x in c for x in ['consulting', 'consult', 'mckinsey', 'bcg', 'bain', 'kpmg', 'pwc', 'ey ', 'ernst', 'grant thornton', 'audit', 'tax']): 
        return 'Consulting'
    return 'Other'
df_analyst['Industry'] = df_analyst['companyName'].apply(get_industry_v2)

print("6. Standardizing Numeric Columns...")
df_analyst['minimumSalary'] = pd.to_numeric(df_analyst['minimumSalary'], errors='coerce')
df_analyst['maximumSalary'] = pd.to_numeric(df_analyst['maximumSalary'], errors='coerce')
df_analyst['minimumExperience'] = pd.to_numeric(df_analyst['minimumExperience'], errors='coerce').fillna(0).astype(int)

df_analyst['midSalary'] = df_analyst.apply(
    lambda row: (row['minimumSalary'] + row['maximumSalary']) / 2 
    if pd.notna(row['minimumSalary']) and pd.notna(row['maximumSalary']) 
    and row['minimumSalary'] > 0 and row['maximumSalary'] > 0
    else None, axis=1
)

print("7. Exporting Clean Data...")
final_cols = [
    'jobId', 'title', 'companyName', 'Industry', 'Clean_City', 
    'tagsAndSkills', 'minimumExperience', 'minimumSalary', 'maximumSalary', 'midSalary'
]

master_path = r"C:\Users\ashis\OneDrive\Desktop\Analyst_Job_Market_Project_2026\3_Cleaned_Data\Master_Analyst_Data_Cleaned.csv"
sample_path_csv = r"C:\Users\ashis\OneDrive\Desktop\Analyst_Job_Market_Project_2026\3_Cleaned_Data\Excel_Analysis_Sample.csv"

df_analyst[final_cols].to_csv(master_path, index=False)

df_fresher = df_analyst[df_analyst['minimumExperience'] <= 1].copy()
df_fresher[final_cols].to_csv(sample_path_csv, index=False)

print("\n" + "="*30)
print("✅ REFINED ETL COMPLETE!")
print(f"Total Analyst Jobs: {len(df_analyst)}")
print(f"Fresher Jobs Saved for Excel: {len(df_fresher)}")
print(f"New Industry Distribution:\n{df_analyst['Industry'].value_counts()}")
print("="*30)

1. Loading 97k dataset... (Wait ~30-60 seconds)
2. Filtering for Analyst roles...
3. Stripping HTML tags...
4. Normalizing Indian City names (RESTORED)...
5. Categorizing Industries (V2)...
6. Standardizing Numeric Columns...
7. Exporting Clean Data...

✅ REFINED ETL COMPLETE!
Total Analyst Jobs: 3809
Fresher Jobs Saved for Excel: 878
New Industry Distribution:
Industry
Other                   1651
IT/Tech                 1517
Staffing/HR Services     224
BFSI                     130
Manufacturing/Core       122
Healthcare                71
Consulting                47
Ecommerce/Logistics       47
Name: count, dtype: int64
